## Module 8, Lecture 2 - Handling Missing Data 

When dealing with real-world data, it is inevitable that some data will be missing. You, as a data scientist or analyst, must grapple with how to handle these missing values. What do they mean? Should they be excluded? Should they be estimated? If estimated, then what approach makes the most sense?  What approach you take to addressing these missing values could impact your final analysis, potentially biasing your estimates, reducing statistical power, or simply leading to invalid or inaccurate conclusions.  

Last week, we learned that one of the critical first steps in an Exploratory Data Analysis (EDA) is examining missing values. Steps we practiced: 

• Quantifying missingness both at the variable and observation level. This is important to evaluate the extent of missing data to determine an imputation approach. If missingness is confined to one variable, should you drop that variable or perhaps seek another dataset that might be more complete? If missingness is confined to certain observations (e.g., specific states or counties), should those be dropped, or would an imputation method allow them to be included, and at what costs to the analysis?  

• Visualizing missingness. We practiced using the naniar library's `vis_miss` function to evaluate whether missing data are systematic or random, which can also provide insights. 

• Understanding missingness. We discussed early in the class that it's always important to understand what missing data mean in a dataset. To recap:  

  - a blank cell: this is 'untidy' data, but it could mean data are missing (e.g., a measurement is missing), or not applicable (N/A). 

  - N/A: could mean it's not applicable (i.e., the measurement doesn't apply to a particular observation)  

  - Missing but should have a value: I would argue that this type of missing is distinct from an observation that might be lacking. Perhaps an observation should have a value but for some reason it's missing and you'd like to account for this (i.e., 'penalize' the data).  

# MCAR, MAR, and MNAR  

In data science parlance (refer to your [reading by van Buuren] (https://stefvanbuuren.name/fimd/sec-MCAR.html)), there are ways we can classify missing data problems:

• Missing Completely at Random (MCAR) - where the probability of data being missing is the same for all cases. One example might be if a few questions on a survey are accidentally skipped by participants due to a printing error that leaves some questions off of a survey. 

• Missing at Random (MAR) - the probability of being missing is the same only within groups of observed data. More general and more realistic than MCAR, where missingness depends on some property or observed variable. For example, a survey where males are less likely to answer a question about their emotional state than females.  

• Missing not at Random (MNAR) - neither MCAR or MAR, where the probability of being missing varies for unknown reasons. For example, a survey where those with weaker opinions about a topic respond less often. This is the most complex type of missingness to handle.  

Now let's use the same Boston weather and crime dataset that we used last week, except this time I've done some manipulation to the data so we can explore the impact of missingness on our analysis.  

### Libraries needed 

We will need tidyverse and ggplot2

In [ ]:
library(tidyverse)
library(ggplot2)

## Data prep

In [ ]:
remove_variable_proportion <- function(data, columns, proportions, seed = 123) {
  set.seed(seed)  # Ensure reproducibility
  
  # Check if columns and proportions match
  if (length(columns) != length(proportions)) {
    stop("The length of 'columns' and 'proportions' must be the same.")
  }
  
  # Loop through each specified column and apply the respective proportion
  for (i in seq_along(columns)) {
    col <- columns[i]
    proportion <- proportions[i]
    
    if (col %in% colnames(data)) {
      # Calculate how many values to remove for this column
      num_to_remove <- floor(proportion * nrow(data))
      
      # Randomly select indices and set them to NA
      if (num_to_remove > 0) {
        indices_to_remove <- sample(1:nrow(data), num_to_remove)
        data[indices_to_remove, col] <- NA
      }
    } else {
      warning(paste("Column", col, "not found in the data frame."))
    }
  }
  
  return(data)
}

### Reading in the Data

This dataset comes from Sommer et al. (2018), "Comparing apples to apples: an environmental criminology analysis of the effects of heat and rain on violent crimes in Boston." 

In [ ]:
data <- read_csv("https://raw.githubusercontent.com/yse-eds-cert/yse-eds-cert-classroom-code-along-notebooks-code_along_notebooks/refs/heads/main/course-3-data-cleaning-and-analysis/module-8/data/weather_crimes_Boston_miss.csv")

#data_rm <- remove_variable_proportion(data, c("TEMP", "TMAX_C", "DEWP", "VISIB", "Wind", "PRCP"), proportions=c(0.1,0.05,0.12,0.07,0.09,0.11), seed=123)

# data_rm %>% write_csv("/data/weather_crimes_Boston_miss.csv")

As a reminder, here's an overview of what these variables stand for (from what I could surmise, as no metadata were included with the data files)  

 - Date: date in YYYY-MM-DD  
  - Day_Wk: day of the week  
  - nb_crimes: number of total crimes  
  - nb... : number of type of crimes  
  - TEMP: temperature in C  
  - TMAX_C: maximum temperature in C  
  - TMIN_C: minimum temperature in C  
  - DEWP: dewpoint  
  - VISIB: visibility  
  - WIND: wind speed  
  - PRCP: precipitation  
  - y20XX: year of data (0/1)  
  - year: year of data  
  - Friday...Weekend: day of week of data (0/1)  
  - _lag: series of lag variables  
  - Season: season of date  
  - Summer...Spring: Season (0/1)  
  - heat.index: heat index  
  - SNOW: whether there was recorded SNOW 

### Exploring Missingness in the Data 

Utilize the steps we practiced last week to explore missingness in our dataset.

For the weather variables, provide a count of how many missing observations there are in TEMP, TMAX_C, TMIN_C, DEWP, VISIB, WIND, and PRCP

In [ ]:
# Overall missing value count
sum(is.na(data))

# Count missing values in weather variables

data %>% summarise(across(TEMP:PRCP,~sum(is.na(.))))

Which of the weather variables has the most missing data? Which has the least?

How might you calculate the percentage of missingness for each variable?

In [ ]:
data %>% summarise(across(TEMP:PRCP,~ round(mean(is.na(.)*100),2)))

Which weather variable has the highest percentage of missingness? Why might it be better to use a percentage of missingness rather than a count?  

## Visualizing missingness

Now utilize the `vis_miss` function from the naniar package to visualize missingness in our weather variables.

In [ ]:
#install.packages("naniar")
library(naniar)

# visualize missingness
vis_miss(data)

How does this picture of missingness look compared to last week's?   

### Evaluating MCAR, MAR and MNAR

What are some ways we can use our tidyverse skills to evaluate whether the missingness is MCAR, MAR or MNAR? Perhaps some weather instruments don't work as well when temperatures are high or when precipitation is great, suggesting we could be dealing with some MNAR missingness.  

Let's try to explore whether we see some systematic missingness due to high temperatures. One way we could do this is to create a new categorical variable called 'Temp_Quantile' to see if we see any systematic missingness at certain temperature thresholds.

In [ ]:
data %>%
  mutate(Temp_Quantile = cut(TEMP, breaks = quantile(TEMP, na.rm = TRUE), include.lowest = TRUE)) %>%
  group_by(Temp_Quantile) %>%
  summarise(across(TEMP:PRCP,~ round(mean(is.na(.)*100),2)), .groups='drop')

Does it seem like there are any patterns of missingness at various temperature ranges in the data?  

Could we use our ggplot skills to plot the missingness proportions to see if data are systematically missing at one temperature range or another? 

In [ ]:
data %>%
  mutate(temp_quantile = cut(TEMP, breaks = unique(quantile(TEMP, na.rm = TRUE), include.lowest = TRUE))) %>%
  group_by(temp_quantile) %>%
  summarise(across(TEMP:PRCP,~ round(mean(is.na(.)*100),2)), .groups='drop') %>%
  pivot_longer(c(TMAX_C:PRCP), names_to="variable", values_to="value") %>%
  filter(!is.na(temp_quantile)) %>%
  ggplot(aes(x=temp_quantile, y=value, group=variable, fill=variable)) +
  geom_col(position="dodge") +
  theme_minimal()

What do you think? Does it look like there is any missingness at a lower or higher temperature range?  

What about for precipitation? Repeat the code snippet above and create a `prcp_quantile` variable to explore:

In [ ]:
data %>%
  mutate(prcp_quantile = cut(PRCP, breaks = unique(quantile(PRCP, na.rm = TRUE), include.lowest = TRUE))) %>%
  group_by(prcp_quantile) %>%
  summarise(across(TEMP:PRCP,~ round(mean(is.na(.)*100),2)), .groups='drop') %>%
  pivot_longer(c(TMAX_C:PRCP), names_to="variable", values_to="value") %>%
  filter(!is.na(prcp_quantile)) %>%
  ggplot(aes(x=prcp_quantile, y=value, group=variable, fill=variable)) +
  geom_col(position="dodge") +
  theme_minimal()

*Work up to here and we will pick up from here for Class*

### Imputing Missing Values  

Now we've established that our missing data are likely random, we should decide what we want to do to address this missingness. You may establish that perhaps imputing a mean or median value for the missing values would be reasonable. Let's try doing this for the missing `TEMP` data. 

Use `mutate` and `case_when` to impute missing TEMP values as the average. Store this new variable as `TEMP_imp`. Use a density plot (hint: `geom_density`) to compare the distribution of TEMP with TEMP_imp (don't forget to pivot_longer!):

In [ ]:
data <- data %>% mutate(TEMP_imp = case_when(is.na(TEMP)~mean(TEMP, na.rm=TRUE)))

# plot
data %>% pivot_longer(c(TEMP, TEMP_imp), names_to="variable", values_to="value") %>%
  ggplot(aes(x=value, fill=variable)) +
  geom_density(alpha=0.5) +
  theme_minimal()

What do you think? Does this imputation work? Why or why not? Was there something else we should've done in our imputation (hint: group_by?)

In [ ]:
data <- data %>% group_by(Season) %>% mutate(TEMP_imp = case_when(is.na(TEMP)~mean(TEMP, na.rm=TRUE)))

# plot
data %>% pivot_longer(c(TEMP, TEMP_imp), names_to="variable", values_to="value") %>%
  ggplot(aes(x=value, fill=variable)) +
  geom_density(alpha=0.5) +
  theme_minimal()

Grouping by season seems to have helped some, but perhaps grouping by month would be even better? Unfortunately our original data doesn't have a month column. Thankfully, we can use the handy `lubridate` package to extract a month out of the Date column in our original data. We can se the `as.Date` function to convert the `Date` column into a data type called `Date` that will tell R to recognize it as a Date. Then we use the `month` function to extract the month.

In [ ]:
library(lubridate) # install.packages("lubridate")

data <- data %>% mutate(Date = as.Date(Date),
    Month = month(Date, label = TRUE, abbr = TRUE) 
  ) %>%
   group_by(Month) %>% mutate(TEMP_imp = case_when(is.na(TEMP)~mean(TEMP, na.rm=TRUE)))

# plot
data %>% pivot_longer(c(TEMP, TEMP_imp), names_to="variable", values_to="value") %>%
  ggplot(aes(x=value, fill=variable)) +
  geom_density(alpha=0.5) +
  theme_minimal()

Bingo! This is the best imputation yet.  

Of course, there are many more sophisticated methods to impute data, and there are many packages that have since been developed to handle missing data and imputation techniques, including machine learning. You can check out some of these on [CRAN](https://cran.r-project.org/web/views/MissingData.html#exploration).  